In [ ]:
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import os
torch.cuda.is_available()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8l.pt')
model.to(device)

In [ ]:
results = model.predict(source="/content/image-building+humain-bus.jpg")

This cell performs visualization of object detection results on a single image.

Steps:
1. Load the image using OpenCV (`cv2.imread`).
2. Convert the image from BGR to RGB because OpenCV loads images in BGR format,
   but Matplotlib expects RGB for correct colors.
3. Loop through the detection results (`results`) from the YOLO model.
   - Extract the bounding box coordinates (`x1, y1, x2, y2`), class index, and confidence score.
   - Draw a rectangle around each detected object using `cv2.rectangle`.
   - Write the class name and confidence score on top of the bounding box using `cv2.putText`.
4. Display the annotated image using Matplotlib (`plt.imshow`) without axes.
   
This allows you to visually inspect which objects were detected and the model’s confidence.

In [ ]:

# Load the image using OpenCV
img_path = "/content/image-building+humain-bus.jpg"
image = cv2.imread(img_path)

# Convert BGR to RGB for displaying with matplotlib
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Loop through the detection results
for res in results:
    for box in res.boxes:
        # Get coordinates
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls = int(box.cls[0])
        conf = float(box.conf[0])

        # Draw rectangle
        cv2.rectangle(image_rgb, (x1, y1), (x2, y2), color=(255, 0, 0), thickness=2)

        # Write class name and confidence
        class_name = model.names[cls]  # assuming you have model.names dict
        label = f"{class_name} {conf:.2f}"
        cv2.putText(image_rgb, label, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

# Display image with boxes
plt.figure(figsize=(12, 8))
plt.imshow(image_rgb)
plt.axis('off')
plt.show()

installs the necessary Python packages for working with CLIP, OpenAI's
Contrastive Language-Image Pretraining model.

Steps:
1. Install `ftfy` and `regex` libraries:
   - `ftfy` fixes text encoding issues.
   - `regex` provides enhanced regular expression support.
2. Install the official OpenAI CLIP library directly from GitHub using pip:
   - This library allows you to compute embeddings for images and text,
     which can be used for similarity comparison, search, or zero-shot classification.

After running this cell, you will be able to use CLIP for feature extraction and
image-text similarity tasks.

In [ ]:
!pip install ftfy regex
!pip install git+https://github.com/openai/CLIP.git

<b>
Sets up the CLIP model for image embedding and defines a function
to compare a query image against a folder of images.

Steps:

1. Import the `clip` library and `numpy`.
2. Load the pre-trained CLIP model ("ViT-B/32") and its corresponding preprocessing
   function for the specified device (`device` can be CPU or GPU).
3. Define the function `preprocess_image(image, folder)`:
   - Converts the input `image` into a CLIP-compatible tensor and computes its embedding.
   - Iterates through all images in the specified `folder`:
       - Loads and preprocesses each image.
       - Computes the CLIP embedding for the database image.
       - Calculates the cosine similarity between the query image and each database image.
       - If the similarity is above a threshold (0.9), it adds the filename and score
         to the results list.
4. Returns a list of images in the folder that are highly similar to the query image.

This allows you to check if a new image already exists in your dataset using
CLIP embeddings and cosine similarity.
</b>

In [ ]:
import clip
import numpy as np

model_clip, preprocess = clip.load("ViT-B/32", device=device)

def preprocess_image(image ,  folder):
  res = []

  query_image = preprocess(image).unsqueeze(0).to(device)

  with torch.no_grad():
      query_feat = model_clip.encode_image(query_image)

  for filename in os.listdir(folder):

        path = os.path.join(folder, filename)

        if os.path.isfile(path):
            print("Processing:", filename)

            db_image = preprocess(Image.open(os.path.join(folder, filename))).unsqueeze(0).to(device)
            with torch.no_grad():
                db_feat = model_clip.encode_image(db_image)

            similarity = torch.cosine_similarity(query_feat, db_feat)
            if similarity > 0.9:
                res.append(f"Found similar image: {filename} with score {similarity.item()}")

  return res

<b>
Performs motion detection on a video, detects persons using YOLO,
checks for new images using CLIP, and records video segments only for new detections.

Steps:

1. Open the video file using OpenCV (`cv2.VideoCapture`) and read the first frame.
2. Convert the first frame to grayscale and apply Gaussian blur for motion detection.
3. Create a folder to store cropped images (`/content/images`) if it doesn't exist.
4. Initialize the VideoWriter parameters (`fourcc`) and variables for recording.
5. Loop through all frames of the video:
    a. Convert the current frame to grayscale and blur it.
    b. Compute the frame difference (`frame_delta`) and threshold it to detect motion.
    c. Count the number of pixels that have changed (`motion_pixels`).
6. If motion is detected:
    a. Run YOLO object detection on the frame.
    b. Loop through detected boxes and check for persons (class 0) with confidence > 0.5.
    c. Crop the detected person from the frame and convert to PIL Image.
    d. Compare the cropped image with the existing folder using `preprocess_image` (CLIP embeddings).
    e. If the person is new (not found in the folder):
        - Start recording the video if not already recording.
        - Write the current frame to the output video.
        - Optionally save the cropped image to the folder.
7. If no motion is detected, skip to the next frame.
8. After finishing, release the video capture and video writer resources.

This workflow ensures that the system only records video segments containing
new persons, reducing unnecessary storage usage and allowing tracking of unique objects.
</b>

In [ ]:
import os

video_path = "/content/drive/MyDrive/men.mp4"
cap = cv2.VideoCapture(video_path)

ret, prev_frame = cap.read()
if not ret:
    raise ValueError(f"Imposible to read this video: {video_path}")

prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
prev_gray = cv2.GaussianBlur(prev_gray, (21, 21), 0)

folder_path = "/content/images"
os.makedirs(folder_path, exist_ok=True)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = None
recording = False

frame_count = 0
image_id = 0

while True:

    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (21, 21), 0)

    frame_delta = cv2.absdiff(prev_gray, gray)
    thresh = cv2.threshold(frame_delta, 25, 255, cv2.THRESH_BINARY)[1]

    motion_pixels = cv2.countNonZero(thresh)

    if motion_pixels > 5000:
      # Motion Detected

        # YOLO detection
        results = model.predict(source=frame)

        for r in results:
            for box in r.boxes:

                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cls = int(box.cls[0])
                conf = float(box.conf[0])

                if cls == 0 and conf > 0.5:

                    cropped = frame[y1:y2, x1:x2]
                    cropped = Image.fromarray(cropped)

                    result_of_exist = preprocess_image(cropped, folder_path)

                    if len(result_of_exist) == 0:

                        if not recording:

                            print("Start Recording")

                            h, w = frame.shape[:2]

                            out = cv2.VideoWriter(
                                "/content/drive/MyDrive/outputRecordv2.mp4",
                                fourcc,
                                20,
                                (w, h)
                            )

                            recording = True

                        if recording:
                            out.write(frame)

                        # cv2.imwrite(f"{folder_path}/image-{image_id}.jpg", cropped)
                        # image_id += 1

    else:
        print("No Motion")

    prev_gray = gray.copy()

    frame_count += 1
    if frame_count == 20:
        break

cap.release()

if out:
    out.release()